# Lunar crater labeling with ipyleaflet

Select a projected lunar GeoTIFF, click a crater center, expand it into a polygon, and export accepted labels to a **GeoPackage**.
Inspired by the interactive map in [NASA eo-validation / ValidationDashboard.ipynb](https://github.com/nasa-nccs-hpda/eo-validation/blob/main/notebooks/ValidationDashboard.ipynb) and its [ValidationDashboard implementation](https://github.com/nasa-nccs-hpda/eo-validation/blob/main/eo_validation/validation_dashboard.py).

The **ipyleaflet** map supports mouse-wheel zoom, drag-to-pan, fullscreen, layer visibility, and draggable polygon vertices. Raster detail refreshes automatically after navigation. Image windows are sent through notebook widgets, so no extra tile-server port or JupyterHub proxy configuration is needed.

### Setup
This notebook requires you to select the 'lfm_kernel_ipyleaflet' kernel from the top right of the webpage (this differs from the kernels used in the other notebooks). 

### Lunar coordinates
The map uses Leaflet's **Simple planar CRS** with locations mapped to **native northing/easting**, not Earth latitude/longitude. The raster's projected lunar CRS remains authoritative; image bounds, clicks, polygons, and output all share that coordinate plane. Geographic rasters must first be projected into an appropriate lunar CRS. There is no Earth basemap, Earth reprojection, or Earth geodesic measurement. Ellipse radii use the raster's native linear units (metres for the supplied example); region-growing limits use source pixels. Rotated rasters are displayed using a north-up VRT in the same CRS.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

repo_root = Path.cwd().parent
repo_root_str = str(repo_root).replace('/panfs/ccds02/nobackup', '/explore/nobackup')
repo_root = Path(repo_root_str)
NOTEBOOK_DIR = repo_root / "notebooks"

if not (repo_root / "lfm").exists():
  raise FileNotFoundError(
      "Cannot find lfm/ directory. Run this notebook from "
      "lfm/notebooks/full_model or update repo_root."
  )

sys.path.insert(0, str(repo_root))

from lfm.data_processing.labeling.craters import CraterLabeler

### Label craters

1. Click **Browse files…** to open the filesystem browser, initially at the configured Explore raster directory. Select a folder and click **Open selected** to enter it. Use **Up one folder**, **Go to folder**, or **Explore default** to navigate, and **Filter name** to find GeoTIFFs in the current folder. Select the desired GeoTIFF and click **Open selected**, then **Load raster**. You can also paste a full path in **Raster**. This browses the notebook server filesystem: on Explore JupyterHub it browses Explore, and in local JupyterLab it browses your local machine. An unavailable Explore path stays selected and shows an error in the browser until you navigate to an accessible folder.
2. **Scroll the mouse wheel to zoom** and **drag the map to pan**. Higher-detail raster windows load automatically after navigation. **Full extent** returns to the overview; **Refresh view** is a manual fallback. The upper-right layer control toggles imagery and labels. Use **Navigate** mode to explore without changing your draft.
3. For the simplest workflow choose **Circle**, click the **geometric center** of a crater, type a number in **Radius**, and press **Enter**. Increasing the number expands the circle from the same center. Radius is in the raster's native units (metres for the example); diameter is twice the radius. Click **Accept crater** when it fits. A tiny initial outline may appear angular at the current zoom—raise its radius. **Ellipse** additionally enables axis ratio and angle.
4. For automatic circle sizing choose **Edge circle** (the default), then click near the geometric center. Or create a manual circle and click **Fit circle to edges**. Set **Min radius / Max radius** in the same native units to bracket the expected rim, then refit. The method fixes your clicked center, evaluates radial contrast over candidate circles in a bounded image window, and chooses a radius with broad angular support. It runs on CPU without model weights. The selected radius populates the numeric field, so you can refine it immediately. Reclick to correct the center. This does not continuously animate expansion and does not deform the result into an irregular outline.
5. Alternatively choose **Region grow**, click inside a crater, and set **Tolerance** in original band units and **Limit px** (maximum 512 source pixels). Click **Regrow / reset** after changing these values. Growth is 4-connected and seed-intensity based, stops at nodata or the radius limit, fills interior holes, and simplifies the outline by 0.75 pixel.
6. Choose **Edit vertices** and drag white handles to fit the rim. Click **Accept crater**. Green outlines are accepted; cyan is the draft. Changing expansion modes or ellipse parameters regenerates the draft and replaces edits. **Discard draft** cancels it; **Undo last crater** removes the most recently accepted label.
7. Every **Accept crater** automatically updates the same file: `<raster_stem>_label_craters.gpkg`, under `LABEL_DIR` (by default the repository’s `notebooks/outputs/labels/`). **Export GeoPackage** updates that file too; no timestamped copies are created. **Load raster** restores saved labels for the selected raster, so you can continue after a restart. Switching rasters changes the label filename automatically; accept or discard the draft first. Writes replace the file atomically only after the new GeoPackage is complete. If autosave fails, the previous file stays intact and your draft remains available for retry. **Undo last crater** also updates the saved file, including an empty layer when the last label is removed. Drafts are not saved. Use one labeling session per raster/output file at a time; concurrent writers are not merged.

**Region growing proposes an outline; it does not detect crater rims.** Shadows and bright terrain may split a crater or connect it to the surroundings. Review each proposal and refine it with vertex edits. A boundary warning means growth may be truncated. Outlines must stay inside raster bounds, but manually drawn outlines may include nodata areas.

Export layer: `craters`. Attributes: `crater_id`, `method`, `source`, `band`, `seed_col`, `seed_row`, `area_native` (squared native CRS units). The GeoPackage preserves the original lunar CRS.

### Choosing an automatic method

**Edge circle** is a lightweight fixed-center radial-edge search. It retains a circular shape and reduces the influence of a single bright arc by combining sectors around the rim. Smoothing, bounded/downsampled windows, and at most 512 candidate radii keep computation bounded. The displayed sector-support fraction is a contrast diagnostic, not a calibrated confidence score. Nodata borders and largely out-of-raster circles are excluded. A fit at a radius limit needs review and usually a wider search range. Shadows, ejecta, neighboring craters, and a misplaced center can still produce the wrong circle.

**Region grow** follows similar pixel intensity and often outlines only the shadowed portion of a crater. It is a separate option from rim-based fitting.

Other possible approaches: [Canny + circular Hough](https://scikit-image.org/docs/0.25.x/auto_examples/edges/plot_circular_elliptical_hough_transform.html) can also search for the center; [active contours](https://scikit-image.org/docs/stable/auto_examples/edges/plot_active_contours.html) deform an initial outline toward edges; [SAM 2](https://github.com/facebookresearch/sam2) supports prompted image segmentation but requires model weights and validation on lunar imagery. Those methods are not implemented here. In particular, a segmentation mask is not guaranteed to represent the geological outer rim.

In [ ]:
# Browse files opens a folder navigator at the default raster directory.
# Use it to navigate the notebook server filesystem and choose a GeoTIFF.
DEFAULT_RASTER = Path("/explore/nobackup/projects/lfm/Benchmarks/Craters/NAC_PHO_E064S3160/NAC_DTM_NEWCRATER6_M1219245090_80CM.TIF")
RASTER_DIR = DEFAULT_RASTER.parent
ROOT = repo_root
LABEL_DIR = NOTEBOOK_DIR / "outputs" / "labels"

# Accepted labels are reloaded from the raster-specific GeoPackage on Load raster.
# Accept or discard the current draft before rerunning this cell.
if "dashboard" in globals():
    dashboard.close()
dashboard = CraterLabeler(data_dir=RASTER_DIR, output_dir=LABEL_DIR,
                          default_raster=DEFAULT_RASTER)
display(dashboard.widget)